In [29]:
import os
import copy
import random
import collections
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Scikit-learn Preprocessing, Splitting, and Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve, auc, brier_score_loss
from sklearn.calibration import calibration_curve, CalibrationDisplay

# Imbalanced Learning Frameworks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# Machine Learning & Deep Learning Frameworks
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Explainable AI (XAI) Libraries
import shap
import lime
import lime.lime_tabular

import preprocess

In [30]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Ensure fully deterministic behavior in PyTorch backends
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [31]:
filepath = "data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
df = preprocess.load_and_clean_dataset(filepath)

#change to a binary label
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

X = df.drop(columns=['Label'])
y = df['Label'].values

print(f"[*] Cleaned Feature matrix shape: {X.shape}")
print(f"[*] Target distribution: Benign (0) = {np.sum(y == 0)}, Attack (1) = {np.sum(y == 1)}")


[*] Loading raw dataset from: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
[*] Dataset Ingestion & Cleaning Audit:
    -> Raw Rows Ingested:          458,968
    -> Empty CSV Padding Purged:   288,602
    -> Valid Network Flows:        170,366
    -> Invalid Flows Dropped (Inf/NaN): 135
    -> Final Usable Flows:         170,231
[*] Cleaned Feature matrix shape: (170231, 78)
[*] Target distribution: Benign (0) = 168051, Attack (1) = 2180


In [32]:
# Step A: Split off 30% of the data into a temporary block, stratifying to preserve class ratios
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Step B: Split the temporary block evenly to yield a 15% Validation set and a 15% Test set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("==================== DATA SPLIT SUMMARY ====================")
print(f"Training Set (70%):   X = {X_train.shape}, y = {y_train.shape}")
print(f"Validation Set (15%): X = {X_val.shape}, y = {y_val.shape}")
print(f"Test Set (15%):       X = {X_test.shape}, y = {y_test.shape}")

==================== DATA SPLIT SUMMARY ====================
Training Set (70%):   X = (119161, 78), y = (119161,)
Validation Set (15%): X = (25535, 78), y = (25535,)
Test Set (15%):       X = (25535, 78), y = (25535,)


In [33]:
# 1. Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# 2. Correlated Feature Groups (Supervisor point 15 & 16)
corr_matrix = X_train_scaled.corr(method="spearman")
# Get upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# Find columns with correlation > 0.90
to_drop = [column for column in upper.columns if any(upper[column].abs() > 0.90)]

print(f"[*] Features identified for potential removal (>0.90 correlation): {len(to_drop)}")

[*] Features identified for potential removal (>0.90 correlation): 39


In [34]:
# Initialize SMOTE-Tomek with a fixed seed for strict reproducibility
smote_tomek = SMOTETomek(random_state=42)

# Resample ONLY the training set
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(
    X_train_scaled,
    y_train
)

print("================= RESAMPLING SUMMARY =================")
print(f"Original Training Class Ratios: 0 = {np.sum(y_train == 0)}, 1 = {np.sum(y_train == 1)}")
print(f"Resampled Training Data Shape:  X = {X_train_resampled.shape}, y = {y_train_resampled.shape}")
print(f"Resampled Training Class Ratios: 0 = {np.sum(y_train_resampled == 0)}, 1 = {np.sum(y_train_resampled == 1)}")

================= RESAMPLING SUMMARY =================
Original Training Class Ratios: 0 = 117635, 1 = 1526
Resampled Training Data Shape:  X = (235254, 78), y = (235254,)
Resampled Training Class Ratios: 0 = 117627, 1 = 117627


In [35]:
print("[*] Initializing XGBoost Classifier...")

# Initialize XGBoost with strict reproducibility parameters
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.1,
    random_state=42,
    use_label_encoder=False,
    early_stopping_rounds=15
)

# Fit model with early stopping monitored against the validation split
xgb_model.fit(
    X_train_resampled, 
    y_train_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

print(f"[*] XGBoost training complete.")
print(f"    -> Best Iteration: {xgb_model.best_iteration}")

[*] Initializing XGBoost Classifier...


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\thesis_env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [16:00:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[*] XGBoost training complete.
    -> Best Iteration: 398


In [36]:
class RobustNetworkSecurityDNN(nn.Module):
    def __init__(self, input_dim):
        super(RobustNetworkSecurityDNN, self).__init__()
        
        # Layer 1: Input to Hidden 1
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.30)
        
        # Layer 2: Hidden 1 to Hidden 2
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.30)
        
        # Layer 3: Hidden 2 to Output Layer
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.sigmoid(self.fc3(x))
        return x

# Set calculation device backend
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_features_count = X_train_resampled.shape[1]

# Instantiate model architecture
model_dnn = RobustNetworkSecurityDNN(input_dim=input_features_count).to(device)
print(f"[*] PyTorch Network mapped successfully onto target hardware device: {device.type.upper()}")

[*] PyTorch Network mapped successfully onto target hardware device: CPU


In [37]:
# Convert DataFrames/Arrays to PyTorch multi-dimensional tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_resampled.values),
    torch.FloatTensor(y_train_resampled).unsqueeze(1)
)
val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)
val_y_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Configure data loader iterations
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

# Set optimizer equations and standard binary loss scoring
criterion = nn.BCELoss()
optimizer = optim.Adam(model_dnn.parameters(), lr=0.001)

# Early Stopping parameters
patience = 10
best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0
max_epochs = 150

print("[*] Initiating DNN optimization loop...")
for epoch in range(1, max_epochs + 1):
    model_dnn.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model_dnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        
    # Validation Evaluation Phase (Zero Leakage Check)
    model_dnn.eval()
    with torch.no_grad():
        val_outputs = model_dnn(val_x_tensor)
        val_loss = criterion(val_outputs, val_y_tensor).item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    
    # Early stopping criteria tracking
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model_dnn.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"[*] Early stopping triggered at Epoch {epoch}. Overfitting threshold neutralized.")
        break

# Roll back parameter matrix states to the optimal captured validation validation loss weights
if best_model_weights is not None:
    model_dnn.load_state_dict(best_model_weights)
print(f"[*] Restored optimal model configurations. Best Validation Loss: {best_val_loss:.5f}")

[*] Initiating DNN optimization loop...
[*] Early stopping triggered at Epoch 21. Overfitting threshold neutralized.
[*] Restored optimal model configurations. Best Validation Loss: 0.04361


In [38]:
# Helper function to generate clean inference probability matrices from our DNN
def get_dnn_probabilities(df_input):
    model_dnn.eval()
    with torch.no_grad():
        tensor_input = torch.FloatTensor(df_input.values).to(device)
        probs = model_dnn(tensor_input).cpu().numpy().flatten()
    return probs

# Step A: Collect raw feature inference array probabilities from validation subsets
xgb_val_probs = xgb_model.predict_proba(X_val_scaled)[:, 1]
dnn_val_probs = get_dnn_probabilities(X_val_scaled)

# Step B: Declare target tuning thresholds range
thresholds_pool = np.arange(0.50, 0.96, 0.05)

best_xgb_threshold = 0.50
best_xgb_f1 = 0.0
best_dnn_threshold = 0.50
best_dnn_f1 = 0.0

print("================= VAL THRESHOLD CALIBRATION =================")
for t in thresholds_pool:
    # Evaluate XGBoost arrays
    xgb_preds = (xgb_val_probs >= t).astype(int)
    xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)
    if xgb_f1 > best_xgb_f1:
        best_xgb_f1 = xgb_f1
        best_xgb_threshold = t
        
    # Evaluate DNN arrays
    dnn_preds = (dnn_val_probs >= t).astype(int)
    dnn_f1 = f1_score(y_val, dnn_preds, zero_division=0)
    if dnn_f1 > best_dnn_f1:
        best_dnn_f1 = dnn_f1
        best_dnn_threshold = t
        
    print(f"Threshold: {t:.2f} | XGB F1: {xgb_f1:.4f} | DNN F1: {dnn_f1:.4f}")

print("\n[*] Calibrated Decision Parameter Options Frozen:")
print(f"    -> Selected Frozen XGBoost Threshold: {best_xgb_threshold:.2f} (Val F1: {best_xgb_f1:.4f})")
print(f"    -> Selected Frozen DNN Threshold:     {best_dnn_threshold:.2f} (Val F1: {best_dnn_f1:.4f})")

================= VAL THRESHOLD CALIBRATION =================
Threshold: 0.50 | XGB F1: 0.9954 | DNN F1: 0.5195
Threshold: 0.55 | XGB F1: 0.9954 | DNN F1: 0.5304
Threshold: 0.60 | XGB F1: 0.9954 | DNN F1: 0.5450
Threshold: 0.65 | XGB F1: 0.9954 | DNN F1: 0.6054
Threshold: 0.70 | XGB F1: 0.9954 | DNN F1: 0.6681
Threshold: 0.75 | XGB F1: 0.9970 | DNN F1: 0.7687
Threshold: 0.80 | XGB F1: 0.9970 | DNN F1: 0.8626
Threshold: 0.85 | XGB F1: 0.9969 | DNN F1: 0.8555
Threshold: 0.90 | XGB F1: 0.9969 | DNN F1: 0.8547
Threshold: 0.95 | XGB F1: 0.9985 | DNN F1: 0.8571

[*] Calibrated Decision Parameter Options Frozen:
    -> Selected Frozen XGBoost Threshold: 0.95 (Val F1: 0.9985)
    -> Selected Frozen DNN Threshold:     0.80 (Val F1: 0.8626)


In [39]:
# Step A: Extract test predictions using frozen configurations
xgb_test_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
dnn_test_probs = get_dnn_probabilities(X_test_scaled)

xgb_test_preds = (xgb_test_probs >= best_xgb_threshold).astype(int)
dnn_test_preds = (dnn_test_probs >= best_dnn_threshold).astype(int)

# Step B: Print formal validation summaries for your thesis report
print("=================== FINAL FROZEN XGBOOST REPORT ===================")
print(f"Applied Decision Threshold: {best_xgb_threshold:.2f}")
print(confusion_matrix(y_test, xgb_test_preds))
print(classification_report(y_test, xgb_test_preds, digits=4))

print("\n==================== FINAL FROZEN DNN REPORT ====================")
print(f"Applied Decision Threshold: {best_dnn_threshold:.2f}")
print(confusion_matrix(y_test, dnn_test_preds))
print(classification_report(y_test, dnn_test_preds, digits=4))

=================== FINAL FROZEN XGBOOST REPORT ===================
Applied Decision Threshold: 0.95
[[25207     1]
 [    4   323]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999     25208
           1     0.9969    0.9878    0.9923       327

    accuracy                         0.9998     25535
   macro avg     0.9984    0.9939    0.9961     25535
weighted avg     0.9998    0.9998    0.9998     25535


==================== FINAL FROZEN DNN REPORT ====================
Applied Decision Threshold: 0.80
[[25126    82]
 [   19   308]]
              precision    recall  f1-score   support

           0     0.9992    0.9967    0.9980     25208
           1     0.7897    0.9419    0.8591       327

    accuracy                         0.9960     25535
   macro avg     0.8945    0.9693    0.9286     25535
weighted avg     0.9966    0.9960    0.9962     25535



In [40]:
# Standardized probability wrappers for the explainers
def xgb_predict_proba_wrapper(x_numpy):
    # Map numpy matrix rows directly back to pandas format to preserve column naming indices natively
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    return xgb_model.predict_proba(df_temp)

def dnn_predict_proba_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    probs_class_1 = get_dnn_probabilities(df_temp)
    probs_class_0 = 1.0 - probs_class_1
    return np.column_stack((probs_class_0, probs_class_1))

# Immediate Sanity Check Test
test_batch = X_val_scaled.iloc[:5].values
xgb_check = xgb_predict_proba_wrapper(test_batch)
dnn_check = dnn_predict_proba_wrapper(test_batch)

print("[*] Prediction wrappers successfully verified:")
print(f"    XGB Output Shape: {xgb_check.shape} | First row: {xgb_check[0]}")
print(f"    DNN Output Shape: {dnn_check.shape} | First row: {dnn_check[0]}")

[*] Prediction wrappers successfully verified:
    XGB Output Shape: (5, 2) | First row: [9.9999994e-01 3.5345582e-08]
    DNN Output Shape: (5, 2) | First row: [1. 0.]


In [41]:

feature_list = X_train_scaled.columns.tolist()
x_train_unresampled_numpy = X_train_scaled.values

# 1. Standard Discretized LIME Explainer (Quartile binning default)
lime_explainer_discrete = lime.lime_tabular.LimeTabularExplainer(
    training_data=x_train_unresampled_numpy,
    feature_names=feature_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    discretize_continuous=True,
    kernel_width=None,
    random_state=42
)

# 2. Continuous LIME Explainer (Direct continuous perturbation without binning)
lime_explainer_continuous = lime.lime_tabular.LimeTabularExplainer(
    training_data=x_train_unresampled_numpy,
    feature_names=feature_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    discretize_continuous=False,
    kernel_width=None,
    random_state=42
)

# Alias default explainer for backward compatibility with downstream code
lime_explainer = lime_explainer_discrete
print("[*] Initialized discrete and continuous LIME explainers successfully.")

[*] Initialized discrete and continuous LIME explainers successfully.


In [42]:

def diagnose_lime_neighborhood(instance, predict_fn, explainer_discrete, explainer_continuous, label_name="Instance", num_samples=1000):
    """
    Extracts the raw perturbed neighborhood generated by LIME and evaluates
    black-box output distribution and local surrogate R^2 under both discretization modes.
    """
    total_features = len(instance)
    
    # 1. Run Discrete LIME
    exp_disc = explainer_discrete.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        labels=(1,),
        num_features=total_features,
        num_samples=num_samples
    )
    
    # 2. Run Continuous LIME
    exp_cont = explainer_continuous.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        labels=(1,),
        num_features=total_features,
        num_samples=num_samples
    )
    
    print(f"\n================ Neighborhood Diagnosis: {label_name} ================")
    print(f"Original Model Prediction [P(Attack)]: {predict_fn(instance.reshape(1, -1))[0, 1]:.5f}")
    print(f"Discrete LIME Local R^2   : {exp_disc.score:.5f}")
    print(f"Continuous LIME Local R^2 : {exp_cont.score:.5f}")
    
    return {
        'discrete_r2': exp_disc.score,
        'continuous_r2': exp_cont.score
    }

# Select one representative Benign sample and one representative Attack sample from validation data
benign_idx = np.where(y_val == 0)[0][0]
attack_idx = np.where(y_val == 1)[0][0]

benign_sample = X_val_scaled.iloc[benign_idx].values
attack_sample = X_val_scaled.iloc[attack_idx].values

print("--- Diagnosing XGBoost Neighborhoods ---")
xgb_benign_diag = diagnose_lime_neighborhood(
    benign_sample, xgb_predict_proba_wrapper, lime_explainer_discrete, lime_explainer_continuous, label_name="Benign Sample (XGB)"
)
xgb_attack_diag = diagnose_lime_neighborhood(
    attack_sample, xgb_predict_proba_wrapper, lime_explainer_discrete, lime_explainer_continuous, label_name="Attack Sample (XGB)"
)

print("\n--- Diagnosing DNN Neighborhoods ---")
dnn_benign_diag = diagnose_lime_neighborhood(
    benign_sample, dnn_predict_proba_wrapper, lime_explainer_discrete, lime_explainer_continuous, label_name="Benign Sample (DNN)"
)
dnn_attack_diag = diagnose_lime_neighborhood(
    attack_sample, dnn_predict_proba_wrapper, lime_explainer_discrete, lime_explainer_continuous, label_name="Attack Sample (DNN)"
)

--- Diagnosing XGBoost Neighborhoods ---

================ Neighborhood Diagnosis: Benign Sample (XGB) ================
Original Model Prediction [P(Attack)]: 0.00000
Discrete LIME Local R^2   : 0.06455
Continuous LIME Local R^2 : 0.09969

================ Neighborhood Diagnosis: Attack Sample (XGB) ================
Original Model Prediction [P(Attack)]: 1.00000
Discrete LIME Local R^2   : 0.20469
Continuous LIME Local R^2 : 0.08700

--- Diagnosing DNN Neighborhoods ---

================ Neighborhood Diagnosis: Benign Sample (DNN) ================
Original Model Prediction [P(Attack)]: 0.00000
Discrete LIME Local R^2   : 0.06081
Continuous LIME Local R^2 : 0.08672

================ Neighborhood Diagnosis: Attack Sample (DNN) ================
Original Model Prediction [P(Attack)]: 0.99635
Discrete LIME Local R^2   : 0.20307
Continuous LIME Local R^2 : 0.12982


In [43]:
# Clear, unresampled background sample for both KernelExplainers
shap_background_baseline = shap.sample(X_train_scaled, 100, random_state=42)

shap_explainer_xgb = shap.KernelExplainer(
    model=xgb_predict_proba_wrapper,
    data=shap_background_baseline
)

shap_explainer_dnn = shap.KernelExplainer(
    model=dnn_predict_proba_wrapper,
    data=shap_background_baseline
)


In [44]:
def extract_aligned_lime_attributions(instance, predict_fn, num_features):
    """
    Generates local linear surrogate attributions and extracts weights directly 
    via feature indices to completely prevent string interval truncation errors.
    """
    # Force explanation extraction explicitly for Class 1 (Web Attack)
    exp = lime_explainer.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        labels=(1,),
        num_features=num_features
    )
    
    # Isolate the underlying index-to-weight mapping array for Class 1
    raw_local_exp = exp.local_exp[1]
    index_to_weight = {feature_idx: weight for feature_idx, weight in raw_local_exp}
    
    # Reconstruct the uniform vector following native column positioning
    total_features = x_train_unresampled_numpy.shape[1]
    lime_vector = np.zeros(total_features)
    for idx in range(total_features):
        lime_vector[idx] = index_to_weight.get(idx, 0.0)
        
    # Capture R-squared local surrogate training fidelity
    fidelity_score = exp.score
    
    return lime_vector, fidelity_score


def extract_aligned_shap_attributions(instance, explainer):
    """
    Extracts local SHAP attribution vectors aligned explicitly to Class 1 (Web Attack)
    natively via unified KernelExplainer multi-output tracking.
    """
    # Unified execution path: sampling permutations with a baseline budget of 100
    raw_values = explainer.shap_values(instance, nsamples=100)
    
    # Handle structural differences across different SHAP / NumPy versions dynamically
    if isinstance(raw_values, list):
        # Format A: List of length 2 -> Index 1 isolates Class 1 (Attack)
        shap_vector = raw_values[1]
    elif isinstance(raw_values, np.ndarray):
        # Format B: NumPy Array -> Inspect dimensions to pull Class 1
        if len(raw_values.shape) == 2:
            if raw_values.shape[1] == 2:   # Shape is (78, 2) -> Column 1 is Class 1
                shap_vector = raw_values[:, 1]
            elif raw_values.shape[0] == 2: # Shape is (2, 78) -> Row 1 is Class 1
                shap_vector = raw_values[1, :]
            else:
                shap_vector = raw_values
        else:
            shap_vector = raw_values
    else:
        shap_vector = raw_values
        
    return np.array(shap_vector).flatten()


In [46]:
#Comprehensive LIME Hyperparameter Search 
def comprehensive_lime_tuning(
    X_train_data,
    X_eval_data,
    y_eval_data,
    predict_fn,
    feature_names,
    kernel_widths=[0.25, 0.75, 1.5, 2.5, None],
    discretize_options=[True, False],
    feature_selectors=['auto', 'highest_weights', 'lasso_path', 'none'],
    samples_per_eval=20,
    random_state=42
):
    """
    Exhaustively searches kernel widths, discretization modes, and feature selection methods.
    Evaluates both Attack and Benign subsets to isolate cohort effects.
    """
    benign_idx = np.where(y_eval_data == 0)[0]
    attack_idx = np.where(y_eval_data == 1)[0]
    
    rng = np.random.default_rng(random_state)
    eval_benign = rng.choice(benign_idx, size=min(samples_per_eval, len(benign_idx)), replace=False)
    eval_attack = rng.choice(attack_idx, size=min(samples_per_eval, len(attack_idx)), replace=False)
    
    total_features = X_train_data.shape[1]
    results = []

    print("--- Starting Systematic LIME Hyperparameter Grid Search ---")
    for disc in discretize_options:
        for kw in kernel_widths:
            for f_select in feature_selectors:
                # Instantiate explainer with the feature selector here
                temp_explainer = lime.lime_tabular.LimeTabularExplainer(
                    training_data=X_train_data.values,
                    feature_names=feature_names,
                    class_names=["Benign", "Web Attack"],
                    mode="classification",
                    discretize_continuous=disc,
                    kernel_width=kw,
                    feature_selection=f_select,
                    random_state=random_state
                )
                
                # 1. Evaluate on Attacks
                atk_scores = []
                for idx in eval_attack:
                    exp = temp_explainer.explain_instance(
                        data_row=X_eval_data.iloc[idx].values,
                        predict_fn=predict_fn,
                        labels=(1,),
                        num_features=total_features,
                        num_samples=1000
                    )
                    atk_scores.append(exp.score)
                    
                # 2. Evaluate on Benign
                ben_scores = []
                for idx in eval_benign:
                    exp = temp_explainer.explain_instance(
                        data_row=X_eval_data.iloc[idx].values,
                        predict_fn=predict_fn,
                        labels=(1,),
                        num_features=total_features,
                        num_samples=1000
                    )
                    ben_scores.append(exp.score)
                    
                mean_atk_r2 = np.mean(atk_scores)
                mean_ben_r2 = np.mean(ben_scores)
                combined_r2 = 0.5 * (mean_atk_r2 + mean_ben_r2) # Balanced evaluation
                
                results.append({
                    'discretize': disc,
                    'kernel_width': str(kw),
                    'feature_selector': f_select,
                    'attack_mean_r2': mean_atk_r2,
                    'benign_mean_r2': mean_ben_r2,
                    'balanced_mean_r2': combined_r2
                })
                
                print(f"Disc: {str(disc):<5} | KW: {str(kw):<5} | Select: {f_select:<15} | "
                      f"Atk R2: {mean_atk_r2:.4f} | Ben R2: {mean_ben_r2:.4f}")

    results_df = pd.DataFrame(results).sort_values(by='balanced_mean_r2', ascending=False)
    
    print("\n================ TOP 5 LIME CONFIGURATIONS ================")
    print(results_df.head(5).to_string(index=False))
    print("===========================================================")
    
    return results_df

# Execute comprehensive parameter search for XGBoost
xgb_lime_grid_df = comprehensive_lime_tuning(
    X_train_data=X_train_scaled,
    X_eval_data=X_val_scaled,
    y_eval_data=y_val,
    predict_fn=xgb_predict_proba_wrapper,
    feature_names=feature_list
)

--- Starting Systematic LIME Hyperparameter Grid Search ---
Disc: True  | KW: 0.25  | Select: auto            | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.25  | Select: highest_weights | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.25  | Select: lasso_path      | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.25  | Select: none            | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.75  | Select: auto            | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.75  | Select: highest_weights | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.75  | Select: lasso_path      | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 0.75  | Select: none            | Atk R2: 0.0000 | Ben R2: 0.0000
Disc: True  | KW: 1.5   | Select: auto            | Atk R2: 0.9635 | Ben R2: 0.0120
Disc: True  | KW: 1.5   | Select: highest_weights | Atk R2: 0.9635 | Ben R2: 0.0120
Disc: True  | KW: 1.5   | Select: lasso_path      | Atk R2: 0.9632 | Ben R2: 0.0120
Disc: True  | KW

In [56]:
# %% [Cell 15d: Diagnostic - Mathematical Proof of Weight Collapse]
def verify_kernel_weight_collapse(instance, kernel_widths=[0.25, 1.5, None]):
    """
    Computes Euclidean distances and resulting exponential kernel weights 
    under continuous Gaussian sampling to test for degenerate weighting.
    """
    total_features = len(instance)
    # Simulate LIME's continuous perturbation (standard normal shift around scaled instance)
    np.random.seed(42)
    perturbations = np.random.normal(0, 1, size=(1000, total_features))
    perturbed_samples = instance + perturbations
    
    # Calculate Euclidean distance in continuous feature space
    distances = np.linalg.norm(perturbed_samples - instance, axis=1)
    
    print(f"Mean Euclidean distance across 1000 perturbed samples: {distances.mean():.4f}")
    print(f"Min distance: {distances.min():.4f} | Max distance: {distances.max():.4f}\n")
    
    for kw in kernel_widths:
        actual_kw = np.sqrt(total_features) * 0.75 if kw is None else kw
        weights = np.exp(-(distances ** 2) / (actual_kw ** 2))
        
        non_zero_count = np.sum(weights > 1e-5)
        max_weight = weights.max()
        mean_weight = weights.mean()
        
        print(f"Kernel Width: {str(kw):<5} (effective kw={actual_kw:.2f})")
        print(f"  -> Samples with non-zero weight (>1e-5): {non_zero_count}/1000")
        print(f"  -> Mean weight: {mean_weight:.2e} | Max weight: {max_weight:.4f}")

sample_instance = X_val_scaled.iloc[0].values
verify_kernel_weight_collapse(sample_instance)

Mean Euclidean distance across 1000 perturbed samples: 8.8128
Min distance: 5.8967 | Max distance: 11.1196

Kernel Width: 0.25  (effective kw=0.25)
  -> Samples with non-zero weight (>1e-5): 0/1000
  -> Mean weight: 2.41e-245 | Max weight: 0.0000
Kernel Width: 1.5   (effective kw=1.50)
  -> Samples with non-zero weight (>1e-5): 0/1000
  -> Mean weight: 2.02e-10 | Max weight: 0.0000
Kernel Width: None  (effective kw=6.62)
  -> Samples with non-zero weight (>1e-5): 1000/1000
  -> Mean weight: 1.75e-01 | Max weight: 0.4527


In [60]:
def evaluate_attack_cohort_agreement(
    X_eval,
    y_eval,
    predict_fn,
    shap_explainer,
    feature_names,
    kernel_width=1.5,
    num_samples=1000,
    eval_sample_size=100,
    random_state=42
):
    """
    Evaluates SHAP-LIME agreement strictly on Web Attack instances where LIME 
    demonstrates strong local surrogate fidelity (R^2 > 0.90).
    """
    # 1. Isolate only Attack samples from validation split
    attack_indices = np.where(y_eval == 1)[0]
    rng = np.random.default_rng(random_state)
    n_samples = min(eval_sample_size, len(attack_indices))
    selected_idx = rng.choice(attack_indices, size=n_samples, replace=False)
    
    eval_subset = X_eval.iloc[selected_idx].reset_index(drop=True)
    total_features = eval_subset.shape[1]
    
    # 2. Instantiate LIME with validated optimal attack configuration
    lime_high_fid_exp = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_scaled.values,
        feature_names=feature_names,
        class_names=["Benign", "Web Attack"],
        mode="classification",
        discretize_continuous=True,
        kernel_width=kernel_width,
        random_state=random_state
    )
    
    records = []
    print(f"--- Running Agreement Analysis on {n_samples} High-Fidelity Attack Instances ---")
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        for idx in range(n_samples):
            instance = eval_subset.iloc[idx].values
            
            # Extract LIME attribution & R^2
            exp = lime_high_fid_exp.explain_instance(
                data_row=instance,
                predict_fn=predict_fn,
                labels=(1,),
                num_features=total_features,
                num_samples=num_samples
            )
            lime_fid = exp.score
            
            raw_local = exp.local_exp[1]
            idx_to_w = {f_idx: w for f_idx, w in raw_local}
            lime_vec = np.array([idx_to_w.get(i, 0.0) for i in range(total_features)])
            
            # Extract SHAP attribution
            shap_vec = extract_aligned_shap_attributions(instance, shap_explainer)
            
            # Similarity metrics
            norm_product = np.linalg.norm(lime_vec) * np.linalg.norm(shap_vec)
            cosine_sim = (np.dot(lime_vec, shap_vec) / norm_product) if norm_product > 0 else 0.0
            
            spearman_corr, _ = stats.spearmanr(lime_vec, shap_vec)
            if np.isnan(spearman_corr):
                spearman_corr = 0.0
                
            records.append({
                'attack_obs_idx': idx,
                'lime_r2': lime_fid,
                'cosine_sim': cosine_sim,
                'spearman_corr': spearman_corr
            })
            
            if (idx + 1) % 25 == 0 or (idx + 1) == n_samples:
                print(f"  Progress: {idx + 1}/{n_samples} evaluations completed.")
                
    return pd.DataFrame(records)

# Run evaluation on XGBoost
xgb_attack_agreement_df = evaluate_attack_cohort_agreement(
    X_eval=X_val_scaled,
    y_eval=y_val,
    predict_fn=xgb_predict_proba_wrapper,
    shap_explainer=shap_explainer_xgb,
    feature_names=feature_list,
    kernel_width=1.5,
    num_samples=1000,
    eval_sample_size=100
)

print("\n================== HIGH-FIDELITY ATTACK AGREEMENT SUMMARY ==================")
print(f"Mean LIME Surrogate Fidelity (R^2): {xgb_attack_agreement_df['lime_r2'].mean():.4f}")
print(f"Median LIME Fidelity (R^2)        : {xgb_attack_agreement_df['lime_r2'].median():.4f}")
print(f"Mean Cosine Similarity            : {xgb_attack_agreement_df['cosine_sim'].mean():.4f}")
print(f"Median Cosine Similarity          : {xgb_attack_agreement_df['cosine_sim'].median():.4f}")
print(f"Mean Spearman Rank Correlation    : {xgb_attack_agreement_df['spearman_corr'].mean():.4f}")
print(f"Median Spearman Rank Correlation  : {xgb_attack_agreement_df['spearman_corr'].median():.4f}")
print("============================================================================")

--- Running Agreement Analysis on 100 High-Fidelity Attack Instances ---
  Progress: 25/100 evaluations completed.
  Progress: 50/100 evaluations completed.
  Progress: 75/100 evaluations completed.
  Progress: 100/100 evaluations completed.

================== HIGH-FIDELITY ATTACK AGREEMENT SUMMARY ==================
Mean LIME Surrogate Fidelity (R^2): 0.9607
Median LIME Fidelity (R^2)        : 0.9629
Mean Cosine Similarity            : 0.3561
Median Cosine Similarity          : 0.3638
Mean Spearman Rank Correlation    : 0.2225
Median Spearman Rank Correlation  : 0.2323


In [61]:
# Systematic Cohort Fidelity Audit (TP, TN, FP, FN)
def evaluate_cohort_fidelity(
    predict_fn,
    model_threshold,
    X_eval,
    y_eval,
    feature_names,
    explainer,
    samples_per_cohort=25,
    random_state=42
):
    """
    Evaluates LIME R^2 fidelity separately across TP, TN, FP, and FN cohorts
    to inspect where the surrogate model succeeds and where it collapses.
    """
    total_features = X_eval.shape[1]
    eval_probs = predict_fn(X_eval.values)[:, 1]
    eval_preds = (eval_probs >= model_threshold).astype(int)
    
    cohort_indices = {
        'True Positive (TP)': np.where((y_eval == 1) & (eval_preds == 1))[0],
        'True Negative (TN)': np.where((y_eval == 0) & (eval_preds == 0))[0],
        'False Positive (FP)': np.where((y_eval == 0) & (eval_preds == 1))[0],
        'False Negative (FN)': np.where((y_eval == 1) & (eval_preds == 0))[0]
    }
    
    rng = np.random.default_rng(random_state)
    records = []
    
    print(f"--- Running Cohort Fidelity Audit (Threshold = {model_threshold:.2f}) ---")
    for cohort_name, indices in cohort_indices.items():
        if len(indices) == 0:
            print(f"[*] Cohort {cohort_name} has 0 observations. Skipping.")
            continue
            
        n_samples = min(samples_per_cohort, len(indices))
        selected = rng.choice(indices, size=n_samples, replace=False)
        print(f"[*] Auditing {cohort_name}: evaluating {n_samples}/{len(indices)} samples...")
        
        for idx in selected:
            instance = X_eval.iloc[idx].values
            exp = explainer.explain_instance(
                data_row=instance,
                predict_fn=predict_fn,
                labels=(1,),
                num_features=total_features,
                num_samples=1000
            )
            records.append({
                'cohort': cohort_name,
                'r2_score': exp.score
            })
            
    cohort_df = pd.DataFrame(records)
    summary_df = cohort_df.groupby('cohort')['r2_score'].agg(['count', 'mean', 'median', 'std']).reset_index()
    
    print("\n======================= COHORT FIDELITY AUDIT REPORT =======================")
    print(summary_df.to_string(index=False))
    print("============================================================================")
    return summary_df

# Evaluate on XGBoost using baseline discrete explainer
xgb_cohort_summary = evaluate_cohort_fidelity(
    predict_fn=xgb_predict_proba_wrapper,
    model_threshold=best_xgb_threshold,
    X_eval=X_val_scaled,
    y_eval=y_val,
    feature_names=feature_list,
    explainer=lime_explainer_discrete
)

--- Running Cohort Fidelity Audit (Threshold = 0.95) ---
[*] Auditing True Positive (TP): evaluating 25/326 samples...
[*] Auditing True Negative (TN): evaluating 25/25208 samples...
[*] Cohort False Positive (FP) has 0 observations. Skipping.
[*] Auditing False Negative (FN): evaluating 1/1 samples...

======================= COHORT FIDELITY AUDIT REPORT =======================
             cohort  count     mean  median      std
False Negative (FN)      1 0.229140 0.22914      NaN
 True Negative (TN)     25 0.086689 0.07775 0.037840
 True Positive (TP)     25 0.201989 0.20197 0.010476


In [53]:
import warnings
from sklearn.exceptions import ConvergenceWarning

def generate_per_observation_evaluations(
    X_eval,
    predict_fn,
    shap_explainer,
    feature_names,
    kernel_width,
    num_samples,
    eval_sample_size=300,
    random_state=42
):
    """
    Runs LIME and SHAP across a representative sample of observations, 
    tracking per-instance fidelity (R2) and vector similarity metrics.
    """
    # 1. Subsample evaluation data to prevent hours of computation
    if len(X_eval) > eval_sample_size:
        print(f"[!] X_eval has {len(X_eval)} rows. Subsampling {eval_sample_size} representative instances for analysis...")
        eval_subset = X_eval.sample(n=eval_sample_size, random_state=random_state).reset_index(drop=True)
    else:
        eval_subset = X_eval.reset_index(drop=True)
        
    # Re-instantiate LIME explainer with selected kernel width
    lime_exp = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_scaled.values,
        feature_names=feature_names,
        class_names=["Benign", "Web Attack"],
        mode="classification",
        kernel_width=kernel_width,
        random_state=random_state
    )
    
    records = []
    total_features = eval_subset.shape[1]
    
    print(f"--- Computing Attributions & Fidelity across {len(eval_subset)} observations ---")
    
    # Context manager to suppress scikit-learn / SHAP singular matrix warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        warnings.filterwarnings("ignore", category=UserWarning, module="shap")
        warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
        
        for idx in range(len(eval_subset)):
            instance = eval_subset.iloc[idx].values
            
            # 1. Extract LIME attribution & per-instance local fidelity R2
            exp = lime_exp.explain_instance(
                data_row=instance,
                predict_fn=predict_fn,
                labels=(1,),
                num_features=total_features,
                num_samples=num_samples
            )
            lime_fid = exp.score  # R2 local surrogate score
            
            raw_local = exp.local_exp[1]
            idx_to_w = {f_idx: w for f_idx, w in raw_local}
            lime_vec = np.array([idx_to_w.get(i, 0.0) for i in range(total_features)])
            
            # 2. Extract SHAP attribution
            shap_vec = extract_aligned_shap_attributions(instance, shap_explainer)
            
            # 3. Compute Similarity / Agreement Metrics
            norm_product = (np.linalg.norm(lime_vec) * np.linalg.norm(shap_vec))
            cosine_sim = (np.dot(lime_vec, shap_vec) / norm_product) if norm_product > 0 else 0.0
            
            spearman_corr, _ = stats.spearmanr(lime_vec, shap_vec)
            if np.isnan(spearman_corr):
                spearman_corr = 0.0
                
            records.append({
                'obs_idx': idx,
                'lime_fidelity_r2': lime_fid,
                'cosine_similarity': cosine_sim,
                'spearman_corr': spearman_corr,
                'lime_vec': lime_vec,
                'shap_vec': shap_vec
            })
            
            # Print progress indicator every 50 samples
            if (idx + 1) % 50 == 0 or (idx + 1) == len(eval_subset):
                print(f"  Progress: {idx + 1}/{len(eval_subset)} observations completed.")
            
    return pd.DataFrame(records)

In [54]:
opt_xgb_kw = best_xgb_params['kernel_width']
opt_xgb_ns = best_xgb_params['num_samples']

# Execute evaluation on a 300-sample subset of validation data
xgb_fidelity_eval_df = generate_per_observation_evaluations(
    X_eval=X_val_scaled,
    predict_fn=xgb_predict_proba_wrapper,
    shap_explainer=shap_explainer_xgb,
    feature_names=feature_list,
    kernel_width=opt_xgb_kw,
    num_samples=opt_xgb_ns,
    eval_sample_size=300
)

[!] X_eval has 25535 rows. Subsampling 300 representative instances for analysis...
--- Computing Attributions & Fidelity across 300 observations ---
  Progress: 50/300 observations completed.
  Progress: 100/300 observations completed.
  Progress: 150/300 observations completed.
  Progress: 200/300 observations completed.
  Progress: 250/300 observations completed.
  Progress: 300/300 observations completed.


In [55]:
def compare_fidelity_stratifications(eval_df, threshold_type='median', custom_threshold=0.3):
    """
    Splits observations into High vs. Low LIME fidelity cohorts 
    and compares LIME vs. SHAP agreement across both groups.
    """
    if threshold_type == 'median':
        split_val = eval_df['lime_fidelity_r2'].median()
        print(f"Stratifying by Median R2 Threshold: {split_val:.4f}\n")
    else:
        split_val = custom_threshold
        print(f"Stratifying by Custom R2 Threshold: {split_val:.4f}\n")
        
    high_fid = eval_df[eval_df['lime_fidelity_r2'] >= split_val]
    low_fid = eval_df[eval_df['lime_fidelity_r2'] < split_val]
    
    summary_data = {
        'Cohort': ['High Fidelity Group', 'Low Fidelity Group'],
        'Count': [len(high_fid), len(low_fid)],
        'Mean LIME R2': [high_fid['lime_fidelity_r2'].mean(), low_fid['lime_fidelity_r2'].mean()],
        'Median LIME R2': [high_fid['lime_fidelity_r2'].median(), low_fid['lime_fidelity_r2'].median()],
        'Mean Cosine Sim': [high_fid['cosine_similarity'].mean(), low_fid['cosine_similarity'].mean()],
        'Mean Spearman Corr': [high_fid['spearman_corr'].mean(), low_fid['spearman_corr'].mean()]
    }
    
    summary_df = pd.DataFrame(summary_data)
    
    print("=================== STRATIFIED AGREEMENT COMPARISON ===================")
    print(summary_df.to_string(index=False))
    print("=======================================================================")
    
    # Hypothesis Verification Check
    diff_cosine = summary_df.loc[0, 'Mean Cosine Sim'] - summary_df.loc[1, 'Mean Cosine Sim']
    diff_spearman = summary_df.loc[0, 'Mean Spearman Corr'] - summary_df.loc[1, 'Mean Spearman Corr']
    
    print("\n[+] Hypothesis Analysis:")
    if diff_cosine > 0 and diff_spearman > 0:
        print("-> Confirmed: High LIME fidelity observations show noticeably stronger agreement with SHAP.")
        print("   This indicates that disagreement in low-fidelity samples is driven by LIME surrogate poor fit.")
    else:
        print("-> Disagreement persists even in high LIME fidelity observations, suggesting fundamental")
        print("   methodological differences (additive global game theory vs. localized ridge surrogate).")

    return summary_df

# Run stratification analysis
stratification_summary = compare_fidelity_stratifications(xgb_fidelity_eval_df, threshold_type='median')

Stratifying by Median R2 Threshold: 0.0796

=================== STRATIFIED AGREEMENT COMPARISON ===================
             Cohort  Count  Mean LIME R2  Median LIME R2  Mean Cosine Sim  Mean Spearman Corr
High Fidelity Group    150      0.138394        0.098240         0.128139            0.077010
 Low Fidelity Group    150      0.061679        0.063367         0.196768            0.121309

[+] Hypothesis Analysis:
-> Disagreement persists even in high LIME fidelity observations, suggesting fundamental
   methodological differences (additive global game theory vs. localized ridge surrogate).
